# 15 - Production Monitoring and Drift

In this notebook, we simulate how a model degrades in production over time due to Data Drift, and how to detect it mathematically.

## Concept Overview
* **Reference Data**: The data the model was trained on.
* **Current Data**: The live data hitting the model in production.
* **Drift Metric**: A statistical test comparing the two distributions (e.g., PSI, K-S test).

## Simulating Data Drift
We will create a 'Training' distribution, and then simulate a sensor slowly breaking in production, shifting the distribution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ks_2samp

np.random.seed(42)

# 1. Training Data (Reference)
reference_data = np.random.normal(loc=50, scale=10, size=1000)

# 2. Production Data (Month 1 - Normal)
prod_month_1 = np.random.normal(loc=50.5, scale=10.2, size=1000)

# 3. Production Data (Month 6 - Drifted due to sensor degradation)
prod_month_6 = np.random.normal(loc=56.0, scale=12.0, size=1000)

# Visualize the drift
plt.figure(figsize=(10, 5))
sns.kdeplot(reference_data, label='Reference (Training)', fill=True, color='blue')
sns.kdeplot(prod_month_1, label='Prod Month 1 (Normal)', fill=True, color='green')
sns.kdeplot(prod_month_6, label='Prod Month 6 (Drifted)', fill=True, color='red')
plt.title('Visualizing Data Drift')
plt.xlabel('Feature Value')
plt.legend()
plt.show()

## Mathematical Detection: Kolmogorov-Smirnov (K-S) Test
We can't just stare at plots all day. We need an automated mathematical trigger to alert us when drift happens.

In [ ]:
def check_drift(reference, production, month_name, threshold=0.05):
    # K-S test checks if two arrays come from the same continuous distribution
    statistic, p_value = ks_2samp(reference, production)
    
    print(f"--- {month_name} ---")
    print(f"P-Value: {p_value:.4f}")
    if p_value < threshold:
        print("🚨 ALERT: Data Drift Detected! Trigger Retraining Pipeline.")
    else:
        print("✅ Status: Normal.")
    print("")

check_drift(reference_data, prod_month_1, "Month 1")
check_drift(reference_data, prod_month_6, "Month 6")

## Modern Tooling: Evidently AI
While we wrote the KS test manually above, the industry standard is to use MLOps libraries like `Evidently` to generate massive HTML dashboards for hundreds of features at once.

In [ ]:
# Note: To run this block, you would need to `pip install evidently`
# import pandas as pd
# from evidently.report import Report
# from evidently.metric_preset import DataDriftPreset

# ref_df = pd.DataFrame({'sensor_value': reference_data})
# prod_df = pd.DataFrame({'sensor_value': prod_month_6})

# report = Report(metrics=[DataDriftPreset()])
# report.run(reference_data=ref_df, current_data=prod_df)
# report.show() # Displays an interactive HTML dashboard inline
print("Evidently AI is the recommended tool for production monitoring pipelines.")

## Industry Notes
Automated retraining pipelines usually have three steps: 1. Detect Drift. 2. Retrain model on last 30 days of data. 3. Shadow Deploy the new model to verify it doesn't crash, then promote to production.

## Exercises
1. Write a function to calculate Population Stability Index (PSI), which is the industry standard for categorical drift.
2. Research 'Concept Drift'. How is it different from the Data Drift shown in this notebook?